In [1]:
import json
import numpy as np
from collections import defaultdict
from datetime import datetime, timezone
from astropy.time import Time
from astropy.coordinates import EarthLocation
import astropy.units as u

In [10]:
def unix_to_lst(unix_t, coords):
    loc = EarthLocation(lon=coords[1]*u.deg, lat=coords[0]*u.deg, height=coords[2]*u.m)
    t_obj = Time(unix_t, format="unix", location=loc)
    lst_time = t_obj.sidereal_time("mean").degree % 360
    return lst_time


def get_sim_passes_lst(json_paths, coords, same_sat = False, fixed_sats = None):
    """
    get the simultaneous passes across pulse_data files for same lst times

    Parameters
    ----------
    json_paths : list of str
        paths to the json pulse_data files. 
    location : astropy.coordinates.EarthLocation
        where the telescope is

    Returns
    -------
    sim_passes: list of dicts
        list of all the passes which happen in two or more files at same time
    """

    # STEP 1: extract all pulses into a dictionary
    interval_bounds = []  
    pulse_dict = {} 

    for path in json_paths:
        with open(path, "r") as f:
            data = json.load(f)

        for interval_str, antenna_data in data.items():
            interval_start = int(interval_str)
            interval_end = interval_start
            for ant_key, ant_dict in antenna_data.items():
                for pulse in ant_dict["pulse_data"]:
                    unix_start = int(interval_start+ pulse["start"])
                    unix_end = int(interval_start + pulse["end"])
                    lst_start = unix_to_lst(unix_start, coords)
                    lst_end = unix_to_lst(unix_end, coords)

                    if unix_end >interval_end:
                        interval_end = int(unix_end)

                    for sat_id in pulse["sats_present"]:
                        snr = pulse['sats_present'][sat_id][2]
                        if unix_start not in pulse_dict:
                            pulse_dict[unix_start] = {
                                "lst": (lst_start, lst_end),
                                "unix": (unix_start, unix_end),
                                "antenna": set(),
                                "sats": set(),
                                "SNR": set(),
                                "file": str(path),
                            }
                        pulse_dict[unix_start]["antenna"].add(ant_key)
                        pulse_dict[unix_start]["sats"].add(int(sat_id))
                        pulse_dict[unix_start]['SNR'].add(snr)

            interval_bounds.append((interval_start, interval_end, path))


    # STEP 2: CHECK NO FILES OVERLAP
    sorted_bounds = sorted(interval_bounds, key=lambda x: x[0])
    for i in range(1, len(sorted_bounds)):
        prev_end = sorted_bounds[i-1][1]
        curr_start = sorted_bounds[i][0]
        if curr_start < prev_end:
            raise ValueError(f"Intervals overlap: {sorted_bounds[i-1]} and {sorted_bounds[i]}")


    #STEP 3: CONVERT PULSE DICTIONARY TO LIST AND SORT
    pulse_entries = []
    for entry in pulse_dict.values():
        entry["antenna"] = list(entry["antenna"])
        entry["sats"] = list(entry["sats"])
        pulse_entries.append(entry)
        
    pulse_entries.sort(key=lambda x: x['lst'][0])  # Sort by LST start

    #STEP 4: GET OVERLAPPING PULSES
    sim_pulses = []
    for i in range(len(pulse_entries)):
        e1 = pulse_entries[i]
        lst1_start, lst1_end = e1['lst']
        group = [e1]

        for j in range(i+1, len(pulse_entries)):
            e2 = pulse_entries[j]
            lst2_start, lst2_end = e2['lst']

            if lst2_start >= lst1_end:
                break  # can't overlap any more, go to next

            sats1, sats2 = set(e1["sats"]), set(e2["sats"])

            if same_sat:
                if not sats1 & sats2:  # no common satellite
                    continue

            if fixed_sats:
                f_sats = set(fixed_sats)
                if not (f_sats & sats1) or not (f_sats & sats2):
                    continue

            if lst2_start < lst1_end and lst2_end > lst1_start:
                group.append(e2)

        if len(group) >= 2:
            sim_pulses.append(group)

    return sim_pulses

In [11]:
paths = [
    '/scratch/thomasb/pulsedata_1753132820_len_67200_1760024247.5361912.json',
    '/scratch/thomasb/pulsedata_1753200150_len_86260_1760835173.json'
]


In [15]:
data = get_sim_passes_lst(paths, 
                          [79.41718333333333, -90.76735, 189], 
                          fixed_sats = [57166, 59051]
                          )

IndexError: list index out of range

In [ ]:
#still need to format the output correctly (depends on what we'll need)
secs_in_lst_day = 86164.091

overlap_pulse = []
for group in data:
    unix_start_1, unix_start_2 = group[0]['unix'], group[1]['unix']
    lstimes1, lstimes2 = group[0]['lst'], group[1]['lst']
    lst_overlap = min(lstimes1[1],lstimes2[1])  - max(lstimes1[0], lstimes2[0])
    lst_overlap = np.round(lst_overlap, decimals=3)
    sec_overlap = secs_in_lst_day / 360 * lst_overlap
    sec_overlap = int(sec_overlap)
    overlap_pulse.append(sec_overlap)

    print('---------------')
    print(unix_start_1, unix_start_2)
    print('lst overlap:', lst_overlap)
    print('sec_overlap:', sec_overlap)

print(np.mean(overlap_pulse))

---------------
(1753277650, 1753277945) (1753191625, 1753192160)
lst overlap: 0.651
sec_overlap: 155
---------------
(1753192640, 1753192955) (1753279040, 1753279545)
lst overlap: 0.33
sec_overlap: 78
---------------
(1753279040, 1753279545) (1753192955, 1753193010)
lst overlap: 0.23
sec_overlap: 55
---------------
(1753283665, 1753283945) (1753197625, 1753198165)
lst overlap: 0.651
sec_overlap: 155
---------------
(1753197625, 1753198165) (1753283945, 1753284130)
lst overlap: 0.773
sec_overlap: 185
---------------
(1753198660, 1753198950) (1753285045, 1753285580)
lst overlap: 0.289
sec_overlap: 69
---------------
(1753285045, 1753285580) (1753198950, 1753199145)
lst overlap: 0.815
sec_overlap: 195
---------------
(1753133705, 1753133885) (1753219885, 1753220445)
lst overlap: 0.686
sec_overlap: 164
---------------
(1753219885, 1753220445) (1753134245, 1753134805)
lst overlap: 0.15
sec_overlap: 35
---------------
(1753222690, 1753222980) (1753136800, 1753137140)
lst overlap: 0.066
sec_

In [14]:
1753192640-1753200150

-7510